In [5]:
import pickle
import numpy as np

# 1. Load data
gt_file_name = "/home/bhoffman/Documents/MT FS24/active-learning-dynamics/goal_traj/forward_helix_v2_goal_trajectory.pkl"
with open(gt_file_name, "rb") as f:
    gt_data = pickle.load(f)

# Extract y, z for a "front view" (discard x)
gt_ee_y = gt_data[..., 1]
gt_ee_z = gt_data[..., 2]

# 2. Compute bounding box in y–z
min_y, max_y = np.min(gt_ee_y), np.max(gt_ee_y)
min_z, max_z = np.min(gt_ee_z), np.max(gt_ee_z)
range_y = max_y - min_y
range_z = max_z - min_z

# 3. Choose a maximum dimension in pixels (e.g., 800 px)
max_dim = 800.0

# Preserve aspect ratio by comparing width vs. height
aspect = range_y / range_z if range_z != 0 else 1.0

if aspect > 1.0:
    # Spiral is "wider" than tall, scale horizontally to max_dim
    svg_width = max_dim
    svg_height = max_dim / aspect
    scale = svg_width / range_y if range_y != 0 else 1.0
else:
    # Spiral is "taller" than wide, scale vertically to max_dim
    svg_height = max_dim
    svg_width = max_dim * aspect
    scale = svg_height / range_z if range_z != 0 else 1.0

# 4. Scale the data into the [0, svg_width/height] coordinate space
#    Also invert z if you want z increasing upwards (typical for math),
#    or skip inversion if you want the "normal" SVG orientation.
points = []
for y_val, z_val in zip(gt_ee_y, gt_ee_z):
    # Shift so that minimum is at 0, then scale
    # For y:
    x_svg = (y_val - min_y) * scale  
    # For z (invert so that min_z -> bottom)
    y_svg = svg_height - ((z_val - min_z) * scale)
    points.append((x_svg, y_svg))

# 5. Create the "d" attribute for the path (Move-to first point, then Line-to each subsequent point)
if len(points) > 0:
    d_cmds = [f"M {points[0][0]},{points[0][1]}"]
    for x, y in points[1:]:
        d_cmds.append(f"L {x},{y}")
    d_attr = " ".join(d_cmds)
else:
    d_attr = ""

# 6. Build the SVG string
svg_str = f"""<svg version="1.1" baseProfile="full"
     width="{svg_width}" height="{svg_height}"
     viewBox="0 0 {svg_width} {svg_height}"
     xmlns="http://www.w3.org/2000/svg">
    <path d="{d_attr}"
        stroke="#489ce4"
        stroke-dasharray="30,30"
        stroke-width="25"
        fill="none" />
</svg>
"""

# 7. Write to file
svg_output_path = "spiral_yz_view.svg"
with open(svg_output_path, "w") as f:
    f.write(svg_str)

print(f"SVG saved to {svg_output_path}")

SVG saved to spiral_yz_view.svg
